<a href="https://colab.research.google.com/github/spookyboogy/whispy/blob/master/test_nb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Whisper + Speaker Diarization

The purpose of this notebook is to provide an easy way to transcribe an audio file using openAI's Whisper, diarize the audio using pyannote/speaker-diarization, and finally merge the two results to create a diarized transcript. 

The benefit of using a colab is that you can use cloud gpu's to achieve faster runtime, but you can also connect to a local runtime to use your own hardware, or clone [the github](https://github.com/spookyboogy/whispy) and run it locally after installing dependencies. 

todo:
  - add gpu option to whispy.py 

# Uploading your audio file

Uploading from google drive

In [1]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
audio_path = '/content/drive/MyDrive/PATH_TO_FILE'
audio_path = '/content/drive/MyDrive/whispy/test/test_0207.wav'

Uploading a local file

In [4]:
import os
cwd = os.getcwd()
ls = os.listdir(cwd)
print(f'\ncwd: {cwd}\ndir: {ls}')


cwd: /content
dir: ['.config', 'drive', 'test_0307.wav', 'sample_data']


# Transcribing with Whisper

Install dependencies

In [ ]:
!pip install ipython-autotime
%load_ext autotime

In [ ]:
!pip install -U git+https://github.com/openai/whisper.git 

In [3]:
import whisper
available_model_list = whisper.available_models()
print("\nModels:\n\t" + '\n\t'.join(available_model_list))

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-jwt31cuv
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-jwt31cuv
  Resolved https://github.com/openai/whisper.git to commit 248b6cb124225dd263bb9bd32d060b6517e067f8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20230314-py3-none-any.whl size=798075 sha256=5c53bb4f55c3681ce36d0418ea27b9a75a275f556338c79d5373c6da6ed1766e
  Stored in directory: /tmp/pip-ephem-wheel-cache-b1o_zi4a/wheels/8b/6c/d0/622666868c179f156cf595c8b6f06f88bc5d80c4b31dccaa03
Successfully built openai-whisper
Models:
	tiny.en
	tiny
	base.en
	base
	small.en
	small
	m

In [4]:
print(f'Loading large-v2 model...')
model = whisper.load_model("large-v2")
print(f'Done loading.')

Loading large-v2 model...


100%|██████████████████████████████████████| 2.87G/2.87G [00:23<00:00, 131MiB/s]


Done loading.
time: 1min 20s (started: 2023-05-20 22:28:26 +00:00)


In [ ]:
options = whisper.DecodingOptions(fp16=False)
print(options)

DecodingOptions(task='transcribe', language=None, temperature=0.0, sample_len=None, best_of=None, beam_size=None, patience=None, length_penalty=None, prompt=None, prefix=None, suppress_tokens='-1', suppress_blank=True, without_timestamps=False, max_initial_timestamp=1.0, fp16=False)


In [ ]:
result = model.transcribe(path, verbose=True)

c:\Users\mattt\AppData\Local\Programs\Python\Python38\lib\site-packages\whisper\transcribe.py:114: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:03.480]  to add to our chat.
[00:03.480 --> 00:04.680]  It's called Stop Motion.
[00:04.680 --> 00:06.080]  Ah, OK.
[00:06.080 --> 00:07.080]  And it's free?
[00:07.080 --> 00:08.320]  We download it from?
[00:08.320 --> 00:10.120]  Yes, it's free.
[00:10.120 --> 00:15.200]  And so we started with a new movie.
[00:15.200 --> 00:19.360]  I don't know if you want to see what's in it.
[00:19.360 --> 00:23.240]  So I put this piece of tape here just
[00:23.240 --> 00:27.560]  to make sure it's always in the same place.
[00:27.560 --> 00:28.240]  And then.
[00:28.240 --> 00:30.160]  Oh, there's the App Store.
[00:30.160 --> 00:31.320]  I don't have an iPhone.
[00:31.320 --> 00:33.960]  Oh, just the App Store?
[00:33.960 --> 00:35.600]  But you can do it with a normal camera.
[00:35.600 --> 00:36.120]  App Store.
[00:36.120 --> 00:36.800]  Yes.
[00:36.800

In [ ]:
for i in result['segments']:
    print(f"{i['id']:3} [{i['start']:7.2f} -> {i['end']:7.2f}] {i['text'].strip()}")

  0 [   0.00 ->    3.48] to add to our chat.
  1 [   3.48 ->    4.68] It's called Stop Motion.
  2 [   4.68 ->    6.08] Ah, OK.
  3 [   6.08 ->    7.08] And it's free?
  4 [   7.08 ->    8.32] We download it from?
  5 [   8.32 ->   10.12] Yes, it's free.
  6 [  10.12 ->   15.20] And so we started with a new movie.
  7 [  15.20 ->   19.36] I don't know if you want to see what's in it.
  8 [  19.36 ->   23.24] So I put this piece of tape here just
  9 [  23.24 ->   27.56] to make sure it's always in the same place.
 10 [  27.56 ->   28.24] And then.
 11 [  28.24 ->   30.16] Oh, there's the App Store.
 12 [  30.16 ->   31.32] I don't have an iPhone.
 13 [  31.32 ->   33.96] Oh, just the App Store?
 14 [  33.96 ->   35.60] But you can do it with a normal camera.
 15 [  35.60 ->   36.12] App Store.
 16 [  36.12 ->   36.80] Yes.
 17 [  36.80 ->   38.60] It probably sent the link to the App Store,
 18 [  38.60 ->   43.40] but you might be able to find it for your store for Android.
 19 [  43.

# New Section